# Ablation: Decoding Strategies

Compares greedy vs sampling (temp 0.1, 0.2, 0.3) on both Italian and English.
Measures quality-diversity tradeoff and abstraction level.

**Memory strategy**: SigExt on CPU -> unloaded -> LLM swapped per temperature.

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!uv pip install -e ../..
from huggingface_hub import login
login()

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.metrics.abstraction import compute_abstraction_score
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import numpy as np

In [ ]:
LANG_CONFIGS = {
    'it': {'preset': 'xlmr-5k-060t', 'quant': '8bit'},
    'en': {'preset': 'xlmr-5k-060t',  'quant': '8bit'},
}
TEMPERATURES = [0.0, 0.1, 0.2, 0.3]

results = {}

for lang, lcfg in LANG_CONFIGS.items():
    print(f'\n{"="*60}\n  {lang.upper()}\n{"="*60}')
    sc = SigExtConfig.from_preset(lang, lcfg['preset'])
    data = get_test_data(lang=lang, num_samples=50, skip_samples=sc.skip_samples)
    sm, st = load_sigext_model(sc.model_id, device='cpu')
    proc = preprocess_dataset(data, sm, st, lang=lang)
    unload_sigext_model(sm, st)

    for temp in TEMPERATURES:
        key = f'{lang}_greedy' if temp == 0 else f'{lang}_temp_{temp}'
        print(f'  -> {key}')
        _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', lcfg['quant'],
                              temperature=temp, do_sample=temp > 0)
        chain = create_summary_chain(pipe, get_summary_prompt(lang, 'source_aware'))
        res = run_inference(proc, chain)
        m = run_evaluation(res, lang=lang)
        m['abstraction'] = {
            'mean': float(np.mean([compute_abstraction_score(r['source'], r['generated_summary']) for r in res]))
        }
        results[key] = m
        clear_gpu_memory()

save_results({'ablation': 'decoding', 'results': results}, 'results/ablation_decoding.json')
print('\nDone!')